In [1]:
import pandas as pd


In [2]:
data_url = "https://raw.githubusercontent.com/fenago/tf/main/Chapter6-Regularization_and_Hyperparameter_Tuning/dataset/shuttle.trn"

data = pd.read_table(data_url, header=None, sep=' ')
data.head()

,0,1,2,3,4,5,6,7,8,9
0,50,21,77,0,28,0,27,48,22,2
1,55,0,92,0,0,26,36,92,56,4
2,53,0,82,0,52,-5,29,30,2,1
3,37,0,76,0,28,18,40,48,8,1
4,37,0,79,0,34,-26,43,46,2,1


In [3]:
y = data.pop(9)
X = data.copy()

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, stratify=y)

In [6]:
!pip install keras-tuner
import keras_tuner as kt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 3.1 MB/s eta 0:00:00


In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Dense

In [8]:
tf.random.set_seed(8)

In [11]:
def model_builder(hp):
    model = tf.keras.Sequential()
    hp_l2 = hp.Choice('l2', values = [0.1, 0.01, 0.001, 0.0001])
    reg_fc1 = Dense(512, input_shape=(9,), activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(l2=hp_l2)) # Changed 'l' to 'l2'
    reg_fc2 = Dense(512, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(l2=hp_l2)) # Changed 'l' to 'l2'
    reg_fc3 = Dense(128, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(l2=hp_l2)) # Changed 'l' to 'l2'
    reg_fc4 = Dense(128, activation='relu',
                    kernel_regularizer=tf.keras.regularizers.l2(l2=hp_l2)) # Changed 'l' to 'l2'
    reg_fc5 = Dense(8, activation='softmax')

    model.add(reg_fc1)
    model.add(reg_fc2)
    model.add(reg_fc3)
    model.add(reg_fc4)
    model.add(reg_fc5)
    loss = tf.keras.losses.SparseCategoricalCrossentropy()
    optimizer = tf.keras.optimizers.Adam(0.001)
    model.compile(optimizer = optimizer, loss = loss, metrics = ['accuracy'])
    return model

In [12]:
tuner = kt.RandomSearch(model_builder, objective='val_accuracy', \
                            max_trials=10)

/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
tuner.search(X_train, y_train, validation_data=(X_test, y_test))

Trial 4 Complete [00h 00m 24s]
val_accuracy: 0.9448275566101074

Best val_accuracy So Far: 0.9838314056396484
Total elapsed time: 00h 01m 19s


In [14]:
best_hps = tuner.get_best_hyperparameters()[0]

In [15]:
best_l2 = best_hps.get('l2')
best_l2

0.0001

In [16]:
model = tuner.hypermodel.build(best_hps)
model.fit(X_train, y_train, epochs=5, \
              validation_data=(X_test, y_test))

Epoch 1/5


/usr/local/lib/python3.10/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


952/952 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.9456 - loss: 0.8312 - val_accuracy: 0.9750 - val_loss: 0.2297
Epoch 2/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.9890 - loss: 0.2904 - val_accuracy: 0.9830 - val_loss: 0.2910
Epoch 3/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - accuracy: 0.9923 - loss: 0.1352 - val_accuracy: 0.9969 - val_loss: 0.0865
Epoch 4/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - accuracy: 0.9955 - loss: 0.1245 - val_accuracy: 0.9943 - val_loss: 0.0896
Epoch 5/5
952/952 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - accuracy: 0.9944 - loss: 0.0899 - val_accuracy: 0.9966 - val_loss: 0.0635
